# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [ ]:
# Your solution goes here

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin
import re


In [2]:
BASE_URL = "http://books.toscrape.com/"

# Rating mapping from words to numbers
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def get_soup(url):
    response = requests.get(url)
    response.encoding = "latin-1"   # helps avoid weird characters like Â
    return BeautifulSoup(response.text, "html.parser")

def parse_rating(tag):
    classes = tag.get("class", [])
    for c in classes:
        if c in RATING_MAP:
            return RATING_MAP[c]
    return None

def get_book_details(book_url):
    soup = get_soup(book_url)

    # Extract UPC, Availability from table
    table = soup.find("table", class_="table table-striped")
    upc = None
    availability = None

    if table:
        for row in table.find_all("tr"):
            header = row.find("th").get_text(strip=True)
            value = row.find("td").get_text(strip=True)

            if header == "UPC":
                upc = value
            if header == "Availability":
                availability = value

    # Extract Genre from breadcrumb
    genre = None
    breadcrumb = soup.find("ul", class_="breadcrumb")
    if breadcrumb:
        items = breadcrumb.find_all("li")
        if len(items) >= 3:
            genre = items[2].get_text(strip=True)

    # Extract Description
    description = None
    desc_header = soup.find("div", id="product_description")
    if desc_header:
        p = desc_header.find_next("p")
        if p:
            description = p.get_text(strip=True)

    return upc, genre, availability, description


def scrape_books(min_rating, max_price):
    all_books = []
    page_url = urljoin(BASE_URL, "catalogue/page-1.html")

    while True:
        soup = get_soup(page_url)
        products = soup.find_all("article", class_="product_pod")

        for prod in products:
            # Title
            title_tag = prod.find("h3").find("a")
            title = title_tag.get("title", "").strip()

            # Book URL
            href = title_tag.get("href")
            book_url = urljoin(BASE_URL, href)

            # Price (with encoding fix)
            price_tag = prod.find("p", class_="price_color")
            price_text = price_tag.get_text(strip=True)

            # Option 1: Clean price by removing ALL non-numeric characters
            price_clean = re.sub(r"[^\d.]", "", price_text)
            price_value = float(price_clean)

            # Rating
            rating_tag = prod.find("p", class_="star-rating")
            rating_value = parse_rating(rating_tag)

            # Filter
            if rating_value is None:
                continue
            if rating_value < min_rating or price_value > max_price:
                continue

            # Details from product page
            upc, genre, availability, description = get_book_details(book_url)

            all_books.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price_value,
                "Rating": rating_value,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

        # Next page
        next_li = soup.find("li", class_="next")
        if next_li:
            next_href = next_li.find("a").get("href")
            page_url = urljoin(page_url, next_href)
        else:
            break

    return pd.DataFrame(all_books)


In [3]:
books_df = scrape_books(min_rating=4, max_price=20)
books_df.head()


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,None,Set Me Free,17.46,5,None,None,None
1,None,The Four Agreements: A Practical Guide to Pers...,17.66,5,None,None,None
2,None,Sophie's World,15.94,5,None,None,None
3,None,Untitled Collection: Sabbath Poems 2014,14.27,4,None,None,None
4,None,This One Summer,19.49,4,None,None,None


In [4]:
books_df.info()
books_df["Genre"].value_counts()
books_df.sort_values("Price (£)").head(10)


<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UPC           0 non-null      object 
 1   Title         75 non-null     str    
 2   Price (£)     75 non-null     float64
 3   Rating        75 non-null     int64  
 4   Genre         0 non-null      object 
 5   Availability  0 non-null      object 
 6   Description   0 non-null      object 
dtypes: float64(1), int64(1), object(4), str(1)
memory usage: 4.2+ KB


,UPC,Title,Price (£),Rating,Genre,Availability,Description
53,None,An Abundance of Katherines,10.00,5,None,None,None
39,None,The Origin of Species,10.01,4,None,None,None
25,None,Greek Mythic History,10.23,5,None,None,None
38,None,History of Beauty,10.29,4,None,None,None
36,None,"NaNo What Now? Finding your editing process, r...",10.41,4,None,None,None
23,None,I Am Pilgrim (Pilgrim #1),10.60,4,None,None,None
63,None,Green Eggs and Ham (Beginner Books B-16),10.79,4,None,None,None
48,None,The Power Greens Cookbook: 140 Delicious Super...,11.05,5,None,None,None
28,None,Dear Mr. Knightley,11.21,5,None,None,None
29,None,City of Fallen Angels (The Mortal Instruments #4),11.23,4,None,None,None
